# EcoHome Energy Advisor - Database Setup

In this notebook, you'll set up the database for the EcoHome Energy Advisor. The database will store:
- Energy usage data (consumption, device types, costs)
- Solar generation data (production, weather conditions)

## Learning Objectives
- Create SQLite database with proper schema
- Populate database with sample data
- Query data for analysis
- Understand database design for energy management


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries
from datetime import datetime, timedelta
import random

from models.energy import DatabaseManager

# Fixed seed so the sample data is reproducible
random.seed(42)

## 2. Initialize Database Manager


In [2]:
# Create a DatabaseManager instance
# Initialize it with the path "data/energy_data.db"
import os

DB_PATH = "data/energy_data.db"

# Start from a fresh database so re-running this notebook doesn't duplicate the sample data
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

db_manager = DatabaseManager(DB_PATH)

## 3. Create Database Tables


In [3]:
# Create the database tables
# Use the create_tables() method from your DatabaseManager
db_manager.create_tables()


Database tables created at data/energy_data.db


## 4. Generate Sample Energy Usage Data


In [4]:
# Generate sample energy usage data for the past 30 days
# Include different device types: EV, HVAC, appliances
# Use realistic consumption patterns (higher during peak hours)
#
# Daily totals are kept realistic for a home with an EV:
# - EV (Tesla Model 3): ~10 kWh/day (a 30-40 mile commute), charged in the evening after work
# - HVAC (central AC): runs harder in the afternoon, ~15-20 kWh/day
# - Appliances (dishwasher, washing machine, dryer): ~1-3 runs per day, mostly in the evening
# Costs use the same time-of-use tariff as the get_electricity_prices tool.

from tools import BASE_PRICE_PER_KWH, PERIOD_MULTIPLIERS, _price_period

appliances = {
    # name: (kWh per run, probability it runs in a given evening hour)
    'Dishwasher': (1.5, 0.25),
    'Washing Machine': (0.5, 0.15),
    'Dryer': (3.0, 0.12),
}
APPLIANCE_HOURS = [19, 20, 21, 22]
EV_CHARGING_HOURS = [18, 19, 20, 21]      # plugs in after getting home
HVAC_PEAK_HOURS = [12, 13, 14, 15, 16, 17]


def rate_at(timestamp):
    period = _price_period(timestamp.hour, timestamp.weekday() >= 5)
    return BASE_PRICE_PER_KWH * PERIOD_MULTIPLIERS[period]


# Generate data for the past 30 days
start_date = datetime.now() - timedelta(days=30)
records_created = 0

for day in range(30):
    current_date = start_date + timedelta(days=day)
    # Daily EV energy need, spread over the evening charging hours (not every day is the same)
    ev_daily_kwh = max(0, random.gauss(10, 3))

    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        usage = []

        # EV: charges only in the evening
        if hour in EV_CHARGING_HOURS:
            usage.append(('EV', 'Tesla Model 3', ev_daily_kwh / len(EV_CHARGING_HOURS) * random.uniform(0.8, 1.2)))

        # HVAC: low baseline, higher in the hot afternoon hours
        hvac_base = 1.5 if hour in HVAC_PEAK_HOURS else 0.5
        usage.append(('HVAC', 'Main AC Unit', max(0, hvac_base + random.uniform(-0.3, 0.3))))

        # Appliances: occasional runs, mostly in the evening
        if hour in APPLIANCE_HOURS:
            for name, (kwh, probability) in appliances.items():
                if random.random() < probability:
                    usage.append(('appliance', name, kwh * random.uniform(0.9, 1.1)))

        for device_type, device_name, consumption in usage:
            # Add record to database
            db_manager.add_usage_record(
                timestamp=timestamp,
                consumption_kwh=consumption,
                device_type=device_type,
                device_name=device_name,
                cost_usd=consumption * rate_at(timestamp)
            )
            records_created += 1

print(f"Created {records_created} energy usage records")

Created 915 energy usage records


## 5. Generate Sample Solar Generation Data


In [5]:
# Generate sample solar generation data
# Consider weather patterns, time of day, and seasonal variations

# Weather conditions and their impact on solar generation
weather_conditions = {
    'sunny': {'multiplier': 1.0, 'probability': 0.4},
    'partly_cloudy': {'multiplier': 0.6, 'probability': 0.3},
    'cloudy': {'multiplier': 0.3, 'probability': 0.2},
    'rainy': {'multiplier': 0.1, 'probability': 0.1}
}

# Generate solar generation data for the same period
start_date = datetime.now() - timedelta(days=30)
generation_records = 0

for day in range(30):
    current_date = start_date + timedelta(days=day)
    
    # Select weather condition for the day
    weather_choice = random.choices(
        list(weather_conditions.keys()),
        weights=[weather_conditions[w]['probability'] for w in weather_conditions.keys()]
    )[0]
    weather_multiplier = weather_conditions[weather_choice]['multiplier']
    
    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        
        # Solar generation only during daylight hours (6 AM to 6 PM)
        if 6 <= hour <= 18:
            # Base generation curve (peak at noon)
            hour_factor = 1 - abs(hour - 12) / 6  # Peak at hour 12
            base_generation = 5.0 * hour_factor  # Max 5 kWh at peak
            
            # Apply weather multiplier
            generation = base_generation * weather_multiplier
            
            # Add some random variation
            generation *= random.uniform(0.8, 1.2)
            generation = max(0, generation)
            
            # Calculate temperature (affects solar efficiency)
            base_temp = 20 + random.uniform(-5, 5)
            temp_factor = 1.0 if 15 <= base_temp <= 35 else 0.9
            
            # Solar irradiance calculation
            irradiance = 800 * hour_factor * weather_multiplier if generation > 0 else 0
            
            # Add record to database
            db_manager.add_generation_record(
                timestamp=timestamp,
                generation_kwh=generation,
                weather_condition=weather_choice,
                temperature_c=base_temp * temp_factor,
                solar_irradiance=irradiance
            )
            generation_records += 1

print(f"Created {generation_records} solar generation records")


Created 390 solar generation records


## 6. Query and Analyze Data


In [6]:
# Query the data to verify it was inserted correctly
# Show total consumption by device type
# Show daily solar generation patterns
# Calculate some basic statistics

# Get recent data for analysis
recent_usage = db_manager.get_recent_usage(24)  # Last 24 hours
recent_generation = db_manager.get_recent_generation(24)

print("=== Energy Usage Analysis ===")
print(f"Total records in last 24 hours: {len(recent_usage)}")

# Group by device type
device_consumption = {}
for record in recent_usage:
    device = record.device_type or 'unknown'
    if device not in device_consumption:
        device_consumption[device] = {'kwh': 0, 'cost': 0, 'records': 0}
    device_consumption[device]['kwh'] += record.consumption_kwh
    device_consumption[device]['cost'] += record.cost_usd or 0
    device_consumption[device]['records'] += 1

print("\nConsumption by device type:")
for device, data in device_consumption.items():
    print(f"  {device}: {data['kwh']:.2f} kWh, ${data['cost']:.2f}, {data['records']} records")

print(f"\n=== Solar Generation Analysis ===")
print(f"Total generation records in last 24 hours: {len(recent_generation)}")

total_generation = sum(r.generation_kwh for r in recent_generation)
print(f"Total generation: {total_generation:.2f} kWh")

# Weather breakdown
weather_breakdown = {}
for record in recent_generation:
    weather = record.weather_condition or 'unknown'
    if weather not in weather_breakdown:
        weather_breakdown[weather] = {'kwh': 0, 'records': 0}
    weather_breakdown[weather]['kwh'] += record.generation_kwh
    weather_breakdown[weather]['records'] += 1

print("\nGeneration by weather condition:")
for weather, data in weather_breakdown.items():
    print(f"  {weather}: {data['kwh']:.2f} kWh, {data['records']} records")


=== Energy Usage Analysis ===
Total records in last 24 hours: 29

Consumption by device type:
  HVAC: 17.31 kWh, $2.91, 22 records
  EV: 10.59 kWh, $2.50, 4 records
  appliance: 3.63 kWh, $0.45, 3 records

=== Solar Generation Analysis ===
Total generation records in last 24 hours: 13
Total generation: 31.99 kWh

Generation by weather condition:
  sunny: 31.99 kWh, 13 records


## 7. Test Database Tools


In [7]:
# Test the database query functions from tools.py
# Import and test: query_energy_usage, query_solar_generation, get_recent_energy_summary

from tools import query_energy_usage, query_solar_generation, get_recent_energy_summary

# Test querying data for the last 7 days
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

print("=== Testing Database Tools ===")
print(f"Querying data from {start_date} to {end_date}")

# Test energy usage query
usage_data = query_energy_usage.invoke(
    input={
        "start_date": start_date, 
        "end_date": end_date,
    }
)
print(f"\nEnergy Usage Query Results:")
print(f"  Total records: {usage_data['total_records']}")
print(f"  Total consumption: {usage_data['total_consumption_kwh']} kWh")
print(f"  Total cost: ${usage_data['total_cost_usd']}")

# Test solar generation query
generation_data = query_solar_generation.invoke(
    input={
        "start_date": start_date, 
        "end_date": end_date,
    }
)
print(f"\nSolar Generation Query Results:")
print(f"  Total records: {generation_data['total_records']}")
print(f"  Total generation: {generation_data['total_generation_kwh']} kWh")
print(f"  Average daily: {generation_data['average_daily_generation']} kWh")

# Test recent energy summary
summary = get_recent_energy_summary.invoke(
    input={
        "hours": 24
    }
)
print(f"\nRecent Energy Summary:")
print(f"  Usage: {summary['usage']['total_consumption_kwh']} kWh, ${summary['usage']['total_cost_usd']}")
print(f"  Generation: {summary['generation']['total_generation_kwh']} kWh")
print(f"  Weather: {summary['generation']['average_weather']}")


=== Testing Database Tools ===
Querying data from 2026-10-21 to 2026-10-28

Energy Usage Query Results:
  Total records: 214
  Total consumption: 225.34 kWh
  Total cost: $39.77

Solar Generation Query Results:
  Total records: 91
  Total generation: 201.7 kWh
  Average daily: 25.21 kWh

Recent Energy Summary:
  Usage: 31.53 kWh, $5.87
  Generation: 31.99 kWh
  Weather: sunny
